# 分布式系统核心理论

> **面试频率**: ⭐⭐⭐⭐⭐  
> **适用岗位**: Senior Data Engineer / Staff Engineer  
> **核心主题**: CAP定理 | 一致性模型 | Leader选举 | 网络分区 | 局部失败处理

---

## 学习目标

1. 理解 CAP 定理在真实数据系统中的体现
2. 区分强一致性与最终一致性的实际影响
3. 掌握 Leader Election 的基本原理
4. 了解网络分区对数据管道的影响及处理策略
5. 掌握局部失败的处理模式

---

## 1. CAP 定理 (高频考点)

### 1.1 CAP 三角定义

CAP 定理由 Eric Brewer 于 2000 年提出，指出分布式系统**最多只能同时保证以下三项中的两项**：

| 属性 | 英文 | 含义 |
|------|------|------|
| **C** | Consistency (一致性) | 每次读操作都能读到最新写入的数据 |
| **A** | Availability (可用性) | 每次请求都能得到响应（不保证是最新数据）|
| **P** | Partition Tolerance (分区容忍性) | 即使网络分区发生，系统仍能运行 |

> **关键洞察**: 网络分区 (P) 在实际分布式系统中**不可避免**，因此真正的选择是 **CP vs AP**。

```
          一致性 (C)
           /\
          /  \
         / CP \   <-- ZooKeeper, HBase
        /      \
       /--------\
      / CA  AP   \
     /            \
    /______________\
可用性(A)        分区容忍(P)
                AP --> Kafka, Cassandra, DynamoDB
```

### 1.2 真实数据系统的 CAP 定位

| 系统 | CAP 定位 | 理由 | 适用场景 |
|------|---------|------|----------|
| **Kafka** | AP | 分区时允许 Leader 继续接受写入，可能造成副本不一致 | 事件流、日志收集 |
| **ZooKeeper** | CP | 分区时拒绝服务而非返回过期数据 | 分布式锁、配置中心 |
| **Cassandra** | AP（可调） | 默认最终一致，但可通过 quorum 调高一致性 | 时序数据、宽表 |
| **Redis Cluster** | CP（读）/ AP（可选）| 默认主节点读写保证一致；可配置从节点读（可能过期）| 缓存、会话 |
| **HBase** | CP | 依赖 ZooKeeper，分区时停服 | OLTP 级别写入 |
| **DynamoDB** | AP（默认）| 最终一致读；强一致读需额外参数 | 全球分布式应用 |

### 1.3 面试决策矩阵

当面试官问`你会选择什么存储系统`时，用这个框架思考：

```
问题: 我的业务能容忍读到过期数据吗？
  |
  +-- 不能 (金融、库存) --> 选 CP 系统 (ZooKeeper, HBase, PostgreSQL)
  |
  +-- 可以 (社交、推荐) --> 选 AP 系统 (Cassandra, DynamoDB, Kafka)
                              再问: 需要高写吞吐?
                                YES --> Cassandra
                                NO  --> DynamoDB (无运维负担)
```

In [ ]:
# CAP 决策矩阵 - Python 模拟
# 根据业务需求推荐合适的存储系统

from dataclasses import dataclass
from typing import List

@dataclass
class StorageSystem:
    name: str
    cap_type: str          # 'CP' or 'AP'
    consistency: str       # 'strong' / 'tunable' / 'eventual'
    write_throughput: str  # 'high' / 'medium' / 'low'
    managed: bool          # fully managed service?
    use_cases: List[str]

STORAGE_SYSTEMS = [
    StorageSystem("ZooKeeper",  "CP", "strong",   "medium", False, ["distributed_lock", "config_center", "leader_election"]),
    StorageSystem("HBase",      "CP", "strong",   "high",   False, ["random_read_write", "time_series"]),
    StorageSystem("Kafka",      "AP", "eventual", "high",   False, ["event_stream", "log_collection", "message_queue"]),
    StorageSystem("Cassandra",  "AP", "tunable",  "high",   False, ["time_series", "wide_table", "geospatial"]),
    StorageSystem("DynamoDB",   "AP", "tunable",  "high",   True,  ["serverless", "gaming", "iot"]),
    StorageSystem("Redis",      "CP", "strong",   "high",   False, ["cache", "session", "real_time_counter"]),
    StorageSystem("PostgreSQL", "CP", "strong",   "medium", False, ["transactional", "financial", "inventory"]),
]

def recommend_storage(
    can_tolerate_stale: bool,
    need_high_write: bool,
    prefer_managed: bool,
    use_case: str
) -> List[StorageSystem]:
    """
    根据业务需求推荐存储系统
    
    Args:
        can_tolerate_stale: 是否能容忍读到过期数据
        need_high_write:    是否需要高写吞吐
        prefer_managed:     是否偏好全托管服务
        use_case:          具体使用场景
    """
    target_cap = "AP" if can_tolerate_stale else "CP"
    candidates = [s for s in STORAGE_SYSTEMS if s.cap_type == target_cap]
    
    # 按写吞吐过滤
    if need_high_write:
        candidates = [s for s in candidates if s.write_throughput == "high"]
    
    # 按托管偏好过滤
    if prefer_managed:
        candidates = [s for s in candidates if s.managed]
    
    # 按使用场景匹配
    matched = [s for s in candidates if use_case in s.use_cases]
    return matched if matched else candidates

# 测试场景 1: 金融账户余额 (不能读到过期数据)
print("=" * 50)
print("场景 1: 金融账户余额查询")
print("需求: 强一致性, 中等写入量, 自建")
results = recommend_storage(
    can_tolerate_stale=False,
    need_high_write=False,
    prefer_managed=False,
    use_case="financial"
)
for s in results:
    print(f"  推荐: {s.name} ({s.cap_type}) - {s.consistency} consistency")

# 测试场景 2: 实时推荐系统 (允许短暂过期)
print("\n" + "=" * 50)
print("场景 2: 实时推荐特征存储")
print("需求: 高写吞吐, 允许最终一致, 偏好托管")
results = recommend_storage(
    can_tolerate_stale=True,
    need_high_write=True,
    prefer_managed=True,
    use_case="iot"  # 类似场景
)
for s in results:
    print(f"  推荐: {s.name} ({s.cap_type}) - write: {s.write_throughput}")

# 测试场景 3: Kafka 消费者 Offset 管理
print("\n" + "=" * 50)
print("场景 3: 分布式锁 / Leader 选举")
results = recommend_storage(
    can_tolerate_stale=False,
    need_high_write=False,
    prefer_managed=False,
    use_case="distributed_lock"
)
for s in results:
    print(f"  推荐: {s.name} ({s.cap_type}) - use cases: {s.use_cases}")

---

## 2. 一致性模型 (高频考点)

### 2.1 一致性级别谱系

从强到弱排列（**越强一致性，性能越差**）：

```
强 |  线性一致性 (Linearizability)
   |      └─ 单操作原子可见，全局时间顺序
   |  顺序一致性 (Sequential Consistency)
   |      └─ 所有进程看到相同操作顺序，但不一定是实时顺序
   |  因果一致性 (Causal Consistency)
   |      └─ 有因果关系的操作保证顺序
   |  读己之写 (Read-your-writes)
   |      └─ 自己写的数据自己一定能读到
   |  单调读 (Monotonic Read)
   |      └─ 读到新值后不会再读到旧值
弱 |  最终一致性 (Eventual Consistency)
       └─ 最终会达到一致，过程中可能读到任意值
```

### 2.2 时序图说明最终一致性问题

```
Client A  写: x=1 --> 主节点
                           |--- 异步复制 ---> 副本1 (x=1) t=100ms
                           |--- 异步复制 ---> 副本2 (x=1) t=200ms

Client B  读: x   --> 副本1 (t=50ms)  返回 x=0  <-- 过期数据!
Client C  读: x   --> 副本2 (t=150ms) 返回 x=0  <-- 过期数据!
Client D  读: x   --> 副本1 (t=200ms) 返回 x=1  <-- 最新数据

时间轴: |----50ms----|----100ms----|----150ms----|----200ms---->
                      副本1更新               副本2更新
```

### 2.3 Read-Your-Writes 问题

用户刚提交表单（写操作），刷新页面却看到旧数据（读到了还没同步的副本）。

**解决方案**:
1. 写后读同一节点（Session Sticky）
2. 写入时记录时间戳，读时带上该时间戳，路由到已同步的副本
3. 写完后强制主节点读（牺牲性能）

In [ ]:
# 模拟最终一致性: 带有延迟传播的多副本系统

import time
import random
import threading
from collections import defaultdict
from datetime import datetime

class Replica:
    """模拟一个数据副本，有随机复制延迟"""
    
    def __init__(self, name: str, replication_delay_ms: float):
        self.name = name
        self.delay = replication_delay_ms / 1000.0
        self._store: dict = {}
        self._lock = threading.Lock()
    
    def write(self, key: str, value, timestamp: float):
        """直接写入（用于主节点）"""
        with self._lock:
            self._store[key] = (value, timestamp)
    
    def replicate(self, key: str, value, timestamp: float):
        """模拟带延迟的异步复制"""
        def _delayed_write():
            time.sleep(self.delay + random.uniform(0, self.delay * 0.3))
            with self._lock:
                # 只接受比当前更新的数据 (防止乱序)
                current = self._store.get(key)
                if current is None or timestamp > current[1]:
                    self._store[key] = (value, timestamp)
        
        t = threading.Thread(target=_delayed_write, daemon=True)
        t.start()
        return t
    
    def read(self, key: str):
        with self._lock:
            entry = self._store.get(key)
            return entry[0] if entry else None

class EventuallyConsistentStore:
    """模拟最终一致性存储（主从架构）"""
    
    def __init__(self):
        self.primary = Replica("primary", replication_delay_ms=0)
        self.replicas = [
            Replica("replica-1", replication_delay_ms=100),
            Replica("replica-2", replication_delay_ms=200),
            Replica("replica-3", replication_delay_ms=50),
        ]
        self._pending_threads = []
    
    def write(self, key: str, value):
        """写入主节点，异步复制到副本"""
        ts = time.time()
        self.primary.write(key, value, ts)
        
        for replica in self.replicas:
            t = replica.replicate(key, value, ts)
            self._pending_threads.append(t)
        
        print(f"[{datetime.now().strftime('%H:%M:%S.%f')[:-3]}] WRITE {key}={value} -> primary")
    
    def read(self, key: str, replica_name: str = "primary"):
        """从指定节点读取"""
        if replica_name == "primary":
            node = self.primary
        else:
            node = next((r for r in self.replicas if r.name == replica_name), self.primary)
        return node.read(key)
    
    def wait_convergence(self):
        """等待所有复制完成（模拟最终一致）"""
        for t in self._pending_threads:
            t.join(timeout=2.0)
        self._pending_threads.clear()

# 演示最终一致性问题
print("=" * 60)
print("演示: 最终一致性 - 写后立即读可能读到过期数据")
print("=" * 60)

store = EventuallyConsistentStore()

# 写入数据
store.write("user:1:balance", 1000)

# 立即从不同节点读取
time.sleep(0.01)  # 10ms 后读取
print("\n10ms 后从各节点读取:")
for node_name in ["primary", "replica-1", "replica-2", "replica-3"]:
    val = store.read("user:1:balance", node_name)
    status = "OK" if val == 1000 else "STALE (过期!)"
    print(f"  {node_name:12s}: {val}  [{status}]")

# 等待收敛
print("\n等待复制完成 (最终一致)...")
store.wait_convergence()
time.sleep(0.05)  # 额外缓冲

print("\n收敛后从各节点读取:")
for node_name in ["primary", "replica-1", "replica-2", "replica-3"]:
    val = store.read("user:1:balance", node_name)
    status = "OK" if val == 1000 else "STALE"
    print(f"  {node_name:12s}: {val}  [{status}]")

print("\n结论: 最终一致性保证'最终'达到一致，但不保证写后立即读到最新值")

---

## 3. Leader Election 基本原理 (重要)

### 3.1 为什么需要 Leader？

在分布式系统中，多个节点同时执行相同任务会导致冲突。Leader Election 确保：
- 只有一个节点负责协调（避免脑裂）
- Leader 失败时自动选举新 Leader
- 保证操作的全局顺序

### 3.2 Bully Algorithm（霸道算法）

```
节点 ID: 1, 2, 3, 4, 5  (ID 最大的成为 Leader)

场景: 节点5 (当前Leader) 崩溃

Step 1: 节点3 发现 Leader 超时
        节点3 --> ELECTION --> 节点4, 节点5

Step 2: 节点4 收到 ELECTION
        节点4 --> OK --> 节点3  (我比你大，我来)
        节点4 --> ELECTION --> 节点5

Step 3: 节点5 没有响应 (已崩溃)
        节点4 等待超时后宣布自己为 Leader
        节点4 --> COORDINATOR --> 节点1, 节点2, 节点3

结果: 节点4 成为新 Leader

时序图:
节点3: [发现超时]--[发ELECTION]--[收到OK]----------[收COORDINATOR]
节点4: ----------------[收ELECTION]--[发OK]--[发ELECTION]--[超时]--[宣布Leader]
节点5: xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx (已崩溃)
```

### 3.3 Raft 共识算法

Raft 是现代系统（etcd、CockroachDB）的主流选择，比 Paxos 更易理解。

**三种角色**:
- **Leader**: 处理所有写请求，发送心跳
- **Follower**: 被动接收，投票
- **Candidate**: 选举期间的临时状态

**选举流程**:
```
1. Follower 超时未收到心跳 → 转为 Candidate
2. Candidate 增加 term，给自己投票，发 RequestVote 给其他节点
3. 多数节点投票 → 成为 Leader
4. Leader 开始发心跳，其他节点重置超时计时器

关键: Election Timeout 随机化 (150ms-300ms)，防止所有节点同时发起选举
```

### 3.4 ZooKeeper 分布式锁

```
实现原理: 基于临时顺序节点 (ephemeral sequential node)

1. 所有客户端在 /locks/resource 下创建顺序节点
   Client A --> /locks/resource/lock-0001
   Client B --> /locks/resource/lock-0002
   Client C --> /locks/resource/lock-0003

2. 序号最小的节点获得锁
   Client A (lock-0001) --> 获得锁

3. 其他客户端 Watch 前一个节点
   Client B watch lock-0001
   Client C watch lock-0002

4. Client A 断开连接 → 临时节点自动删除 → Client B 收到通知 → 获得锁
```

In [ ]:
# 模拟 Bully Algorithm 和简化版 Raft 选举

import random
import time
from enum import Enum
from typing import Optional, List, Dict

class NodeState(Enum):
    FOLLOWER  = "Follower"
    CANDIDATE = "Candidate"
    LEADER    = "Leader"
    DEAD      = "Dead"

class RaftNode:
    """简化版 Raft 节点模拟"""
    
    def __init__(self, node_id: int, cluster: 'RaftCluster'):
        self.node_id = node_id
        self.cluster = cluster
        self.state = NodeState.FOLLOWER
        self.current_term = 0
        self.voted_for: Optional[int] = None
        self.votes_received = 0
        # 随机选举超时 (150-300ms 范围，这里用 1-3 轮)
        self.election_timeout = random.randint(1, 3)
        self.heartbeat_count = 0
    
    def receive_heartbeat(self, leader_term: int, leader_id: int):
        if leader_term >= self.current_term:
            self.current_term = leader_term
            self.state = NodeState.FOLLOWER
            self.heartbeat_count = 0
            self.voted_for = None
    
    def tick(self) -> Optional[str]:
        """模拟一个时间 tick"""
        if self.state == NodeState.DEAD:
            return None
        
        if self.state == NodeState.LEADER:
            # Leader 发送心跳
            for node in self.cluster.nodes:
                if node.node_id != self.node_id and node.state != NodeState.DEAD:
                    node.receive_heartbeat(self.current_term, self.node_id)
            return f"Node-{self.node_id} [LEADER] 发送心跳 (term={self.current_term})"
        
        if self.state == NodeState.FOLLOWER:
            self.heartbeat_count += 1
            if self.heartbeat_count >= self.election_timeout:
                # 超时，发起选举
                return self._start_election()
        
        return None
    
    def _start_election(self) -> str:
        self.state = NodeState.CANDIDATE
        self.current_term += 1
        self.voted_for = self.node_id
        self.votes_received = 1  # 给自己投票
        
        msg = f"Node-{self.node_id} 发起选举 (term={self.current_term})\n"
        
        # 请求其他节点投票
        for node in self.cluster.nodes:
            if node.node_id != self.node_id and node.state != NodeState.DEAD:
                vote_granted = node.request_vote(self.current_term, self.node_id)
                if vote_granted:
                    self.votes_received += 1
                    msg += f"  Node-{node.node_id} 投票给 Node-{self.node_id}\n"
        
        majority = len([n for n in self.cluster.nodes if n.state != NodeState.DEAD]) // 2 + 1
        
        if self.votes_received >= majority:
            self.state = NodeState.LEADER
            self.cluster.current_leader = self.node_id
            msg += f"  => Node-{self.node_id} 获得 {self.votes_received} 票 (需要 {majority})，成为 LEADER!"
        else:
            self.state = NodeState.FOLLOWER
            msg += f"  => Node-{self.node_id} 仅获得 {self.votes_received} 票，选举失败"
        
        return msg
    
    def request_vote(self, candidate_term: int, candidate_id: int) -> bool:
        if candidate_term > self.current_term:
            self.current_term = candidate_term
            self.voted_for = candidate_id
            self.state = NodeState.FOLLOWER
            return True
        return False

class RaftCluster:
    def __init__(self, num_nodes: int):
        self.nodes: List[RaftNode] = [
            RaftNode(i, self) for i in range(1, num_nodes + 1)
        ]
        self.current_leader: Optional[int] = None
        # 先手动选一个 Leader
        self.nodes[0].state = NodeState.LEADER
        self.nodes[0].current_term = 1
        self.current_leader = 1
    
    def kill_node(self, node_id: int):
        for node in self.nodes:
            if node.node_id == node_id:
                node.state = NodeState.DEAD
                if self.current_leader == node_id:
                    self.current_leader = None
                print(f"[KILLED] Node-{node_id} 已崩溃")
                break
    
    def run_ticks(self, num_ticks: int):
        for tick in range(num_ticks):
            print(f"\n--- Tick {tick + 1} ---")
            for node in self.nodes:
                result = node.tick()
                if result:
                    print(result)
            
            leader_nodes = [n for n in self.nodes if n.state == NodeState.LEADER]
            if len(leader_nodes) > 1:
                print(f"WARNING: 脑裂! 有 {len(leader_nodes)} 个 Leader!")
    
    def status(self):
        print("\n集群状态:")
        for node in self.nodes:
            print(f"  Node-{node.node_id}: {node.state.value:10s} term={node.current_term}")

# 演示
print("=" * 60)
print("Raft Leader Election 模拟")
print("=" * 60)

cluster = RaftCluster(num_nodes=5)
print("\n初始集群状态 (Node-1 为初始 Leader):")
cluster.status()

print("\n运行 2 个 tick (正常心跳):")
cluster.run_ticks(2)

print("\n" + "=" * 40)
print("模拟 Leader 崩溃...")
cluster.kill_node(1)

print("\n崩溃后运行 tick (触发新选举):")
cluster.run_ticks(3)

cluster.status()

---

## 4. Network Partition 影响 (高频考点)

### 4.1 网络分区对数据管道的影响

```
正常状态:
  Producer → [Kafka Broker 1, 2, 3] → Consumer Group

网络分区发生:
  [机房A: Broker 1, 2] ≠≠≠≠≠ [机房B: Broker 3]
  
  Producer (在机房A) → Broker 1, 2 继续接收消息
  Consumer (在机房B) → 只看到 Broker 3 (落后!)
  
  问题:
  1. Broker 3 可能成为 Leader (如果原 Leader 在机房A)
  2. 脑裂: 两个机房各自选出 Leader
  3. 分区结束后: 消息需要协调合并
```

### 4.2 Kafka 中的脑裂问题

```
Kafka 的解决方案:
1. ISR (In-Sync Replicas): 只有在 ISR 中的副本才能成为 Leader
2. min.insync.replicas: 写入至少 N 个副本才算成功
3. 选举时需要 Controller 参与 (ZooKeeper/KRaft 管理)

Fencing Token 防止脑裂:
  旧 Leader (token=1) 尝试写入 → Storage 拒绝 (token < 当前 token=2)
  新 Leader (token=2) 写入    → Storage 接受
```

### 4.3 处理策略

| 策略 | 说明 | 适用场景 |
|------|------|----------|
| **Quorum Write** | 写入 > N/2 个节点才算成功 | 强一致性要求 |
| **Fencing Token** | 单调递增令牌，防止旧 Leader 写入 | 分布式锁 |
| **Epoch Number** | 每次 Leader 变更递增 Epoch | Kafka ISR |
| **Wait-for-repair** | 分区恢复前拒绝写入 | CP 系统 (ZooKeeper) |
| **Last-write-wins** | 冲突时以时间戳最新的为准 | AP 系统 (Cassandra) |

In [ ]:
# 模拟 Quorum Write (多数派写入)

import random
from typing import List, Tuple

class QuorumCluster:
    """模拟基于 Quorum 的写入策略"""
    
    def __init__(self, n_nodes: int, write_quorum: int, read_quorum: int):
        self.n = n_nodes
        self.w = write_quorum   # 写入需要的确认数
        self.r = read_quorum    # 读取需要的确认数
        self.nodes = {i: {} for i in range(n_nodes)}  # node_id -> data store
        print(f"集群配置: N={n_nodes}, W={write_quorum}, R={read_quorum}")
        print(f"W + R > N? {write_quorum + read_quorum} > {n_nodes}: {write_quorum + read_quorum > n_nodes}")
        if write_quorum + read_quorum > n_nodes:
            print("=> 强一致性: 每次读取必然包含最新写入的节点")
        else:
            print("=> 可能读到过期数据")
    
    def write(self, key: str, value, available_nodes: List[int]) -> Tuple[bool, int]:
        """
        写入数据到可用节点
        
        Returns: (success, acks_received)
        """
        acks = 0
        for node_id in available_nodes:
            if node_id in self.nodes:
                self.nodes[node_id][key] = value
                acks += 1
        
        success = acks >= self.w
        return success, acks
    
    def read(self, key: str, available_nodes: List[int]):
        """
        从可用节点读取 (读取 R 个节点，返回最新值)
        """
        if len(available_nodes) < self.r:
            return None, "insufficient_nodes"
        
        values = []
        for node_id in available_nodes[:self.r]:
            val = self.nodes[node_id].get(key)
            values.append((node_id, val))
        
        # 取最新值 (简化: 假设非None的值更新)
        non_null = [v for _, v in values if v is not None]
        result = non_null[-1] if non_null else None
        return result, values

print("=" * 60)
print("场景 1: 强一致性配置 (N=3, W=2, R=2)")
print("=" * 60)
cluster = QuorumCluster(n_nodes=3, write_quorum=2, read_quorum=2)

# 正常写入 (3个节点都可用)
print("\n写入 x=100 到节点 [0, 1, 2]:")
ok, acks = cluster.write("x", 100, available_nodes=[0, 1, 2])
print(f"  写入{'成功' if ok else '失败'} (确认数={acks}, 需要W={cluster.w})")

# 网络分区: 节点2 不可用，写入 x=200 到节点 [0, 1]
print("\n网络分区! 节点2 不可用")
print("写入 x=200 到节点 [0, 1]:")
ok, acks = cluster.write("x", 200, available_nodes=[0, 1])
print(f"  写入{'成功' if ok else '失败'} (确认数={acks}, 需要W={cluster.w})")

# 从不同节点子集读取
print("\n读取测试:")
for read_nodes in [[0, 1], [1, 2], [0, 2]]:
    val, details = cluster.read("x", available_nodes=read_nodes)
    print(f"  从节点 {read_nodes} 读取: x={val}  详情={details}")

print("\n" + "=" * 60)
print("场景 2: 分区时写入失败 (W=2, 只有1个节点可用)")
print("=" * 60)
cluster2 = QuorumCluster(n_nodes=3, write_quorum=2, read_quorum=2)
cluster2.write("x", 100, available_nodes=[0, 1, 2])

print("\n严重分区: 只有节点0可用")
ok, acks = cluster2.write("x", 999, available_nodes=[0])
print(f"写入 x=999 {'成功' if ok else '失败'} (确认数={acks}, 需要W={cluster2.w})")
print("=> CP 系统正确行为: 拒绝写入而不是接受不一致的写入")

---

## 5. Partial Failure 处理 (高频考点)

### 5.1 分布式计算的 8 个谬误

Peter Deutsch 提出的新手常见错误假设：

```
1. 网络是可靠的              (实际: 丢包、超时、分区)
2. 延迟为零                  (实际: 有延迟且不稳定)
3. 带宽无限                  (实际: 带宽有限且共享)
4. 网络是安全的              (实际: 需要认证加密)
5. 拓扑不会改变              (实际: 节点增减、IP变化)
6. 只有一个管理员            (实际: 多团队，配置冲突)
7. 传输代价为零              (实际: 跨区/跨云有成本)
8. 网络是同构的              (实际: 混合云、不同协议)
```

### 5.2 失败模式

| 失败类型 | 描述 | 检测难度 | 例子 |
|---------|------|---------|------|
| **Crash-stop** | 节点崩溃后永不恢复 | 容易 (超时) | 进程被 kill |
| **Crash-recovery** | 节点崩溃后重启 | 中等 | 服务器重启 |
| **Omission** | 某些消息丢失 | 困难 | 网络丢包 |
| **Byzantine** | 节点行为任意（可能恶意）| 非常困难 | 数据损坏、恶意节点 |

### 5.3 Circuit Breaker (断路器) 模式

```
状态机:
  
  CLOSED (正常) ──[失败次数 >= 阈值]──> OPEN (断开)
     ^                                      |
     |                               [超时后]
     |                                      v
     +──[测试请求成功]────────── HALF-OPEN (试探)
                                      |
                               [测试失败]
                                      v
                                   OPEN

CLOSED: 正常转发请求
OPEN:   直接快速失败，不调用下游 (保护下游, 快速响应)
HALF-OPEN: 允许少量请求通过，测试下游是否恢复
```

In [ ]:
# Python 实现 Circuit Breaker (断路器)

import time
import random
from enum import Enum
from typing import Callable, Any, Optional
from functools import wraps

class CircuitState(Enum):
    CLOSED    = "CLOSED"     # 正常
    OPEN      = "OPEN"       # 断开
    HALF_OPEN = "HALF_OPEN"  # 试探

class CircuitBreakerError(Exception):
    """断路器开启时抛出"""
    pass

class CircuitBreaker:
    """
    断路器实现
    
    保护下游服务免受级联失败的影响:
    - 失败超过阈值 -> 打开断路器 (快速失败)
    - 超时后 -> 进入半开状态
    - 半开测试成功 -> 关闭断路器
    """
    
    def __init__(
        self,
        name: str,
        failure_threshold: int = 3,
        recovery_timeout: float = 5.0,
        half_open_max_calls: int = 1
    ):
        self.name = name
        self.failure_threshold = failure_threshold
        self.recovery_timeout = recovery_timeout
        self.half_open_max_calls = half_open_max_calls
        
        self._state = CircuitState.CLOSED
        self._failure_count = 0
        self._last_failure_time: Optional[float] = None
        self._half_open_calls = 0
    
    @property
    def state(self) -> CircuitState:
        if self._state == CircuitState.OPEN:
            # 检查是否应该进入半开状态
            if (self._last_failure_time and
                time.time() - self._last_failure_time > self.recovery_timeout):
                self._state = CircuitState.HALF_OPEN
                self._half_open_calls = 0
                print(f"[{self.name}] 进入 HALF_OPEN 状态，开始试探")
        return self._state
    
    def call(self, func: Callable, *args, **kwargs) -> Any:
        """通过断路器调用函数"""
        current_state = self.state
        
        if current_state == CircuitState.OPEN:
            raise CircuitBreakerError(
                f"断路器 [{self.name}] 已打开，快速失败 (Fast Fail)"
            )
        
        if current_state == CircuitState.HALF_OPEN:
            if self._half_open_calls >= self.half_open_max_calls:
                raise CircuitBreakerError(
                    f"断路器 [{self.name}] 半开状态，已达到最大试探次数"
                )
            self._half_open_calls += 1
        
        try:
            result = func(*args, **kwargs)
            self._on_success()
            return result
        except Exception as e:
            self._on_failure()
            raise
    
    def _on_success(self):
        if self._state == CircuitState.HALF_OPEN:
            print(f"[{self.name}] 试探成功，断路器关闭")
        self._state = CircuitState.CLOSED
        self._failure_count = 0
    
    def _on_failure(self):
        self._failure_count += 1
        self._last_failure_time = time.time()
        
        if self._failure_count >= self.failure_threshold:
            if self._state != CircuitState.OPEN:
                print(f"[{self.name}] 失败次数={self._failure_count}，断路器打开!")
            self._state = CircuitState.OPEN
    
    def __repr__(self):
        return f"CircuitBreaker({self.name}, state={self.state.value}, failures={self._failure_count})"

# 模拟下游服务 (随机失败)
call_count = 0

def unreliable_api_call(fail_mode: bool = False) -> str:
    global call_count
    call_count += 1
    
    if fail_mode:
        raise ConnectionError("下游服务不可用")
    return f"成功响应 #{call_count}"

# 演示断路器行为
print("=" * 60)
print("Circuit Breaker 演示")
print("=" * 60)

cb = CircuitBreaker(
    name="payment-service",
    failure_threshold=3,
    recovery_timeout=2.0  # 2秒后进入半开
)

# Phase 1: 正常调用
print("\nPhase 1: 正常调用")
for i in range(2):
    try:
        result = cb.call(unreliable_api_call, fail_mode=False)
        print(f"  调用 {i+1}: {result} | {cb}")
    except Exception as e:
        print(f"  调用 {i+1} 失败: {e}")

# Phase 2: 下游服务崩溃 (连续失败)
print("\nPhase 2: 下游服务崩溃")
for i in range(5):
    try:
        result = cb.call(unreliable_api_call, fail_mode=True)
    except CircuitBreakerError as e:
        print(f"  调用 {i+1}: [断路器] {e}")
    except ConnectionError as e:
        print(f"  调用 {i+1}: [连接错误] {e} | {cb}")

# Phase 3: 等待恢复超时
print("\nPhase 3: 等待 2 秒后进入半开状态...")
time.sleep(2.1)

# Phase 4: 下游服务恢复
print("\nPhase 4: 下游服务恢复，试探调用")
try:
    result = cb.call(unreliable_api_call, fail_mode=False)
    print(f"  试探调用: {result} | {cb}")
except Exception as e:
    print(f"  试探调用失败: {e}")

print(f"\n最终状态: {cb}")

### 5.4 Timeout 设计原则

```
超时设计的黄金法则:

1. 始终设置超时 (Never trust network calls without timeout)
   requests.get(url, timeout=5)  # 5秒超时

2. 超时时间参考 P99 延迟
   P50 = 100ms, P99 = 800ms -> 设置 timeout = 1500ms

3. 区分 Connection Timeout 和 Read Timeout
   Connect: 3s (建立连接)
   Read:    30s (等待响应)

4. 级联超时预算
   API Gateway: 10s
    └── Service A: 8s
         └── Database: 5s
         └── Cache: 1s
   
   每层留缓冲，防止子调用超时导致父调用也超时

5. Deadline Propagation
   gRPC 的 context.WithDeadline() 会传递剩余时间
   避免子调用超时时间超过父调用剩余时间
```

---

## 复习要点

### CAP 定理
- P (分区容忍) 在现实中不可避免，真正的选择是 **CP vs AP**
- Kafka = AP，ZooKeeper = CP，Cassandra = AP (tunable)，Redis Cluster = CP
- 选择存储系统时，先问`能容忍过期数据吗？`

### 一致性模型
- 强一致性 (线性一致) → 高延迟，适合金融
- 最终一致性 → 低延迟，可能读到过期数据，适合社交/推荐
- Read-your-writes 问题通过 Session Sticky 或时间戳路由解决

### Leader Election
- Bully: 最大 ID 获胜，简单但网络开销大
- Raft: 随机超时防止同时选举，多数派投票，现代系统首选
- ZooKeeper: 临时顺序节点实现分布式锁

### 网络分区
- 分区时面临 CP/AP 取舍
- Quorum Write (W + R > N) 保证强一致读
- Fencing Token 防止脑裂时旧 Leader 的写入

### 局部失败
- 8 个谬误: 不要假设网络可靠、延迟为零
- Circuit Breaker: CLOSED → OPEN → HALF-OPEN → CLOSED
- 始终设置超时，用 P99 延迟作为参考基准

---

## 练习

### 练习 1: CAP 系统分类

为以下系统选择 CAP 分类并说明理由：
1. MySQL 主从复制（从库读）
2. etcd（Kubernetes 配置存储）
3. Amazon S3
4. MongoDB (默认配置)
5. Kafka (min.insync.replicas=2, acks=all)

### 练习 2: 场景设计

你负责设计一个**实时库存管理系统**，需要：
- 库存扣减必须准确（不能超卖）
- 高并发写入（每秒 10,000 次）
- 读取可以有 100ms 延迟

请选择合适的 CAP 策略，并说明技术选型。

### 练习 3: Quorum 计算

一个 5 节点集群，计算以下配置的特性：
1. W=3, R=3: 能保证强一致性吗？能容忍几个节点失败？
2. W=1, R=1: 写入性能如何？一致性如何？
3. W=5, R=1: 这个配置的优缺点？

In [ ]:
# 练习 3: Quorum 分析工具
# 完成 analyze_quorum 函数，输出各配置的特性

def analyze_quorum(n: int, w: int, r: int) -> dict:
    """
    分析 Quorum 配置
    
    TODO: 实现以下计算:
    - strong_consistency: W + R > N?
    - write_fault_tolerance: 允许几个写节点失败?
    - read_fault_tolerance: 允许几个读节点失败?
    - write_availability: 写入成功概率 (假设每个节点可用率 0.99)
    """
    # 你的代码
    pass

# 测试
configs = [(5, 3, 3), (5, 1, 1), (5, 5, 1), (3, 2, 2)]
for n, w, r in configs:
    result = analyze_quorum(n, w, r)
    print(f"N={n}, W={w}, R={r}: {result}")

### 练习 4: Circuit Breaker 扩展

扩展上面的 `CircuitBreaker` 类，添加：
1. **成功率阈值**: 不仅看绝对失败次数，还看失败率（如最近 10 次中失败率 > 50%）
2. **Metrics 收集**: 记录 `total_calls`, `success_calls`, `failure_calls`, `rejected_calls`
3. **fallback**: 断路器打开时执行备用逻辑（如返回缓存值）

### 练习 5: 网络分区场景分析

你的 Kafka 集群有 3 个 Broker，`min.insync.replicas=2`，`acks=all`。
机房 A 有 Broker 1,2，机房 B 有 Broker 3。

发生网络分区时：
1. Producer 在机房 A，向 Partition Leader (Broker 1) 写入，会成功吗？为什么？
2. Consumer 在机房 B，能消费到最新消息吗？
3. 分区恢复后，Broker 3 需要做什么才能重新加入 ISR？
4. 如何配置才能在分区时宁可拒绝写入而不接受不一致数据？